# Classificação Supervisionada com KMeans no Dataset Adult (Balanceado)

Este notebook apresenta um pipeline **didático e numerado** para classificação supervisionada utilizando KMeans no dataset Adult, **com balanceamento das classes** via oversampling. O objetivo é comparar o desempenho do agrupamento com e sem desbalanceamento, detalhando cada etapa do processo.


A seguir, cada etapa do fluxo é numerada e explicada de forma didática:


1. **Importação das Bibliotecas**
   - Importação de todas as bibliotecas necessárias para manipulação de dados, visualização, pré-processamento e clustering.
2. **Carregamento e Balanceamento dos Dados**
   - Carregamento do dataset Adult e aplicação do balanceamento das classes por oversampling, igualando a quantidade de exemplos das duas classes para evitar viés do modelo.
3. **Pré-processamento dos Dados**
   - Codificação das variáveis categóricas, normalização dos dados e visualização da distribuição da variável alvo. Garante que o KMeans funcione corretamente, pois é sensível à escala dos dados.
4. **Divisão em Treino/Teste**
   - Separação dos dados em conjuntos de treino e teste de forma estratificada, preservando a proporção das classes.
5. **Definição do Número de Clusters**
   - Definição do número de clusters a ser utilizado pelo KMeans, com base em conhecimento prévio ou método do cotovelo.
6. **Implementação do KMeans Supervisionado**
   - Criação de uma classe que associa cada cluster ao rótulo mais frequente no treino, permitindo avaliação supervisionada.
7. **Treinamento e Avaliação Inicial**
   - Treinamento do modelo no conjunto de treino e avaliação inicial no conjunto de teste, incluindo visualização da matriz de confusão.
8. **Visualização dos Agrupamentos com PCA**
   - Redução de dimensionalidade dos dados para 2D usando PCA e visualização dos agrupamentos e centros dos clusters em gráfico de dispersão.
9. **Repetição dos Experimentos e Análise Estatística**
   - Execução do experimento múltiplas vezes com diferentes seeds, cálculo de métricas agregadas (média, desvio padrão, matriz de confusão média) e geração de gráficos para análise de robustez.
10. **Análise dos Resultados e Métricas de Avaliação**
    - Discussão dos resultados, comparação com o dataset original, explicação das métricas (acurácia, precisão, recall, especificidade, F1-score) e interpretação da matriz de confusão média.


Cada seção do notebook está identificada e explicada para facilitar o entendimento do pipeline completo de classificação com KMeans em dados balanceados.

Verdadeiro Negativo (TN):
Quantidade de exemplos negativos corretamente classificados como negativos.
Exemplo: Pessoas que realmente ganham <=50K e foram classificadas como <=50K.

Falso Positivo (FP):
Quantidade de exemplos negativos classificados incorretamente como positivos.
Exemplo: Pessoas que ganham <=50K, mas foram classificadas como >50K (falso alarme).

Falso Negativo (FN):
Quantidade de exemplos positivos classificados incorretamente como negativos.
Exemplo: Pessoas que ganham >50K, mas foram classificadas como <=50K (erro de omissão).

Verdadeiro Positivo (TP):
Quantidade de exemplos positivos corretamente classificados como positivos.
Exemplo: Pessoas que realmente ganham >50K e foram classificadas como >50K.

Acurácia:
Proporção de acertos (positivos e negativos) sobre o total de exemplos.
Fórmula: (TP + TN) / (TP + TN + FP + FN)
Traduz a taxa geral de acerto do modelo.

Precisão (Precision):
Proporção de positivos previstos que realmente são positivos.
Fórmula: TP / (TP + FP)
Traduz o quanto o modelo é confiável quando prevê a classe positiva.

Revocação (Recall) ou Sensibilidade:
Proporção de positivos reais que foram corretamente identificados.
Fórmula: TP / (TP + FN)
Traduz a capacidade do modelo de encontrar todos os positivos.

Especificidade:
Proporção de negativos reais corretamente identificados.
Fórmula: TN / (TN + FP)
Traduz a capacidade do modelo de identificar corretamente os negativos.

F1-Score:
Média harmônica entre precisão e revocação.
Fórmula: 2 * (Precisão * Revocação) / (Precisão + Revocação)
Traduz o equilíbrio entre precisão e revocação, útil quando há desbalanceamento.

## 1. Importação das Bibliotecas

Discussão dos resultados, comparação com o dataset original, explicação das métricas (acurácia, precisão, recall, especificidade, F1-score) e interpretação da matriz de confusão média.

In [ ]:
# ===============================
# 1. Importação das Bibliotecas
# ===============================
# Importa todas as bibliotecas necessárias para manipulação de dados, visualização, pré-processamento e clustering.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import resample
from sklearn.cluster import KMeans
import os
import seaborn as sns
from sklearn.decomposition import PCA

# Define o diretório para salvar imagens
img_dir = 'img'
os.makedirs(img_dir, exist_ok=True)

## 2. Carregamento e Balanceamento dos Dados

O dataset Adult é carregado e a classe minoritária (>50K) é balanceada por oversampling, igualando a quantidade de exemplos das duas classes. Isso evita que o modelo favoreça a classe majoritária.

In [ ]:
# Carrega o dataset Adult, define nomes das colunas e transforma a variável alvo em binária
adult_df_raw = pd.read_csv('data/AdultDataset/adult.data', header=None, na_values=' ?', skipinitialspace=True)
adult_df_raw.columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
    'hours-per-week', 'native-country', 'income'
]
# income_bin: 1 se >50K, 0 caso contrário
adult_df_raw['income_bin'] = adult_df_raw['income'].apply(lambda val: 1 if val.strip() == '>50K' else 0)

# Remove linhas com valores ausentes
adult_df = adult_df_raw.dropna().copy()

# Separa as classes majoritária e minoritária
# Classe 0: <=50K (maioria), Classe 1: >50K (minoria)
df_majority = adult_df[adult_df['income_bin'] == 0]
df_minority = adult_df[adult_df['income_bin'] == 1]

# Oversampling: replica exemplos da minoria até igualar a maioria
n_majority = len(df_majority)
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=n_majority,
    random_state=42
)

# Junta as duas classes e embaralha
adult_df_balanced = pd.concat([df_majority, df_minority_upsampled])
adult_df_balanced = adult_df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Exibe a nova distribuição das classes
print('Distribuição após balanceamento:')
print(adult_df_balanced['income_bin'].value_counts())

## 3. Pré-processamento dos Dados

As variáveis categóricas são codificadas numericamente, os dados são normalizados e a distribuição da variável alvo é visualizada. O pré-processamento garante que o KMeans funcione corretamente, pois o algoritmo é sensível à escala dos dados.

In [ ]:
# Codifica variáveis categóricas (exceto a coluna alvo original 'income')
for col in adult_df_balanced.select_dtypes(include='object').columns:
    if col != 'income':
        adult_df_balanced[col] = LabelEncoder().fit_transform(adult_df_balanced[col].astype(str))

# Separa features e alvo binário
y = adult_df_balanced['income_bin'].values
X = adult_df_balanced.drop(['income', 'income_bin'], axis=1).values

# Normaliza os dados para média 0 e desvio padrão 1
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Visualiza a distribuição da variável alvo após balanceamento
plt.figure(figsize=(5,3))
sns.countplot(x=y)
plt.title('Distribuição da coluna income (binária) após balanceamento')
plt.xticks([0,1],["<=50K",">50K"])
plt.show()

## 4. Divisão em Treino/Teste

Os dados balanceados são divididos em conjuntos de treino e teste de forma estratificada, preservando a proporção das classes.

In [ ]:
# Divide o dataset balanceado em treino (70%) e teste (30%) de forma estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

## 5. Definição do Número de Clusters

Definimos o número de clusters a ser utilizado pelo KMeans, com base em conhecimento prévio ou método do cotovelo.

In [ ]:
# Define o número de clusters (ajuste conforme o método do cotovelo ou conhecimento prévio)
n_clusters = 10

## 6. Implementação do KMeans Supervisionado

Criação de uma classe que associa cada cluster ao rótulo mais frequente no treino, permitindo avaliação supervisionada.

In [ ]:
# Classe para KMeans supervisionado: associa cada cluster ao rótulo mais frequente no treino
class KMeansSupervisionado:
    def __init__(self, n_clusters=n_clusters, random_state=0):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
        self.cluster_labels_ = None

    def fit(self, X, y):
        # Ajusta o KMeans e associa cada cluster ao rótulo mais frequente
        clusters = self.kmeans.fit_predict(X)
        self.cluster_labels_ = []
        for i in range(self.n_clusters):
            mask = (clusters == i)
            if np.any(mask):
                label = np.bincount(y[mask]).argmax()
            else:
                label = -1
            self.cluster_labels_.append(label)

    def predict(self, X):
        # Prediz o cluster e converte para o rótulo associado
        clusters = self.kmeans.predict(X)
        return np.array([self.cluster_labels_[c] for c in clusters])

    def evaluate(self, X, y_true):
        # Avalia acurácia e matriz de confusão
        y_pred = self.predict(X)
        acc = accuracy_score(y_true, y_pred)
        cm = confusion_matrix(y_true, y_pred)
        return acc, cm

## 7. Treinamento e Avaliação Inicial

Treinamento do modelo no conjunto de treino e avaliação inicial no conjunto de teste, incluindo visualização da matriz de confusão.

In [ ]:
# Treina o KMeans supervisionado e avalia no conjunto de teste
clf = KMeansSupervisionado(n_clusters=n_clusters, random_state=30)
clf.fit(X_train, y_train)
acc, cm = clf.evaluate(X_test, y_test)
print(f'Acurácia: {acc:.4f}')
print('Matriz de Confusão:')
print(cm)

# Visualiza a matriz de confusão
plt.figure(figsize=(6,5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Matriz de Confusão - KMeans (Adult Balanceado)')
plt.colorbar()
plt.ylabel('Verdadeiro')
plt.xlabel('Predito')
plt.savefig(f'{img_dir}/kmeans_adult_balance_confusion_matrix.png')
plt.show()

## 8. Visualização dos Agrupamentos com PCA

Redução de dimensionalidade dos dados para 2D usando PCA e visualização dos agrupamentos e centros dos clusters em gráfico de dispersão.

In [ ]:
# Redução de dimensionalidade para 2D com PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Ajusta o KMeans nos dados completos para obter os clusters e centros
cls = KMeans(n_clusters=n_clusters, random_state=42)
clusters = cls.fit_predict(X)
centros = cls.cluster_centers_
centros_pca = pca.transform(centros)
acc, cm = clf.evaluate(X_test, y_test)

print(f'Acurácia: {acc:.4f}')
print('Matriz de Confusão:')
print(cm)
plt.figure(figsize=(6,5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Matriz de Confusão - KMeans (Adult Balanceado)')
plt.colorbar()
plt.ylabel('Verdadeiro')
plt.xlabel('Predito')
plt.savefig(f'{img_dir}/kmeans_adult_balance_confusion_matrix.png')
plt.show()

# Gráfico 2D dos clusters e centros (X em preto)
plt.figure(figsize=(10,7))
plt.scatter(
    X_pca[:,0], X_pca[:,1], 
    c=clusters, cmap='tab10', alpha=0.35, s=18, edgecolor='none', label='Amostras'
)
plt.scatter(
    centros_pca[:,0], centros_pca[:,1], 
    c='black', marker='X', s=220, linewidths=2.5, zorder=4, label='Centros (X)'
)
plt.title('Visualização dos Clusters e Centros (PCA) - KMeans (Adult Balanceado)', fontsize=15, fontweight='bold')
plt.xlabel('Componente Principal 1', fontsize=12)
plt.ylabel('Componente Principal 2', fontsize=12)
plt.grid(alpha=0.25)
plt.legend(frameon=False, fontsize=12, loc='best')
plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'kmeans_adult_balance_clusters_pca.png'), dpi=150)
plt.show()

## 9. Repetição dos Experimentos e Análise Estatística

Execução do experimento múltiplas vezes com diferentes seeds, cálculo de métricas agregadas (média, desvio padrão, matriz de confusão média) e geração de gráficos para análise de robustez.

In [ ]:
# ===============================
# 6. Repetição dos Experimentos e Análise Estatística
# ===============================
# Nesta etapa, executamos o KMeans supervisionado 30 vezes, cada vez com uma semente diferente, para avaliar a robustez do método.
# Para cada repetição, dividimos os dados em treino e teste, treinamos o modelo, prevemos os rótulos e calculamos as métricas de desempenho.
# Ao final, calculamos a média e o desvio padrão das acurácias e do erro quadrático médio (MSE), além da matriz de confusão média.
# Também geramos gráficos para visualizar a variação das métricas ao longo das execuções e salvamos todos os resultados para análise posterior.

from sklearn.metrics import ConfusionMatrixDisplay

acuracias = []  # Lista para armazenar a acurácia de cada repetição
mse_list = []   # Lista para armazenar o MSE de cada repetição
matrizes_confusao = []  # Lista para armazenar a matriz de confusão de cada repetição

for seed in range(1, 31):
    # 6.1. Para cada semente, divide os dados de forma estratificada
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y)

    # 6.2. Treina o KMeans supervisionado com a semente atual
    clf = KMeansSupervisionado(n_clusters=n_clusters, random_state=seed)
    clf.fit(X_train, y_train)

    # 6.3. Realiza a predição no conjunto de teste
    y_pred = clf.predict(X_test)

    # 6.4. Calcula as métricas de avaliação
    acc = accuracy_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    # 6.5. Armazena os resultados
    acuracias.append(acc)
    mse_list.append(mse)
    matrizes_confusao.append(confusion_matrix(y_test, y_pred))

# 6.6. Converte listas para arrays numpy para facilitar o cálculo estatístico
acuracias = np.array(acuracias)
mse_array = np.array(mse_list)
matrizes_confusao = np.array(matrizes_confusao)

# 6.7. Calcula a matriz de confusão média (média dos valores de cada célula ao longo das execuções)
matriz_confusao_media = np.mean(matrizes_confusao, axis=0)
matriz_confusao_media_int = np.round(matriz_confusao_media).astype(int)

# 6.8. Exibe as métricas agregadas para análise geral do desempenho
print(f'Acurácia média: {acuracias.mean():.4f}')
print(f'Desvio padrão da acurácia: {acuracias.std():.4f}')
print(f'MSE médio: {mse_array.mean():.4f}')
print(f'Desvio padrão do MSE: {mse_array.std():.4f}')
print('\nMatriz de Confusão Média (30 execuções):')
print(matriz_confusao_media_int)

# 6.9. Gráfico de acurácia por repetição
plt.figure(figsize=(7,4))
plt.plot(range(1, 31), acuracias, marker='o', color='tab:blue')
plt.xlabel('Repetição')
plt.ylabel('Acurácia')
plt.title('Acurácia por repetição - KMeans (Adult Balanceado)')
plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'kmeans_adult_balance_accuracy_repetitions.png'))
plt.show()

# 6.10. Gráfico de MSE por repetição
plt.figure(figsize=(7,4))
plt.plot(range(1, 31), mse_array, marker='o', color='tab:red')
plt.xlabel('Repetição')
plt.ylabel('Erro Médio Quadrático (MSE)')
plt.title('MSE por repetição - KMeans (Adult Balanceado)')
plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'kmeans_adult_balance_mse_repetitions.png'))
plt.show()

# 6.11. Visualização gráfica da matriz de confusão média
# Útil para identificar padrões de acerto/erro recorrentes ao longo das execuções
disp = ConfusionMatrixDisplay(confusion_matrix=matriz_confusao_media_int)
disp.plot(cmap=plt.cm.Blues)
plt.title('Matriz de Confusão Média - KMeans (Adult Balanceado)')
plt.tight_layout()
plt.savefig(os.path.join(img_dir, 'kmeans_adult_balance_confusion_matrix_media.png'))
plt.show()

# 6.12. Salva os resultados das execuções para análise posterior (npy e csv)
np.save(os.path.join(img_dir, 'kmeans_adult_balance_accuracies.npy'), acuracias)
np.savetxt(os.path.join(img_dir, 'kmeans_adult_balance_accuracies.csv'), acuracias, delimiter=',')
np.save(os.path.join(img_dir, 'kmeans_adult_balance_mse_repetitions.npy'), mse_array)
np.savetxt(os.path.join(img_dir, 'kmeans_adult_balance_mse_repetitions.csv'), mse_array, delimiter=',')
np.save(os.path.join(img_dir, 'kmeans_adult_balance_confusion_matrix_media.npy'), matriz_confusao_media)
np.savetxt(os.path.join(img_dir, 'kmeans_adult_balance_confusion_matrix_media.csv'), matriz_confusao_media, delimiter=',')

# 6.13. Cálculo detalhado das métricas binárias (apenas para problemas 2x2)
if matriz_confusao_media_int.shape == (2, 2):
    TN, FP, FN, TP = matriz_confusao_media_int[0,0], matriz_confusao_media_int[0,1], matriz_confusao_media_int[1,0], matriz_confusao_media_int[1,1]
    total = TN + FP + FN + TP
    acuracia = (TP + TN) / total if total > 0 else 0
    precisao = TP / (TP + FP) if (TP + FP) > 0 else 0
    revocacao = TP / (TP + FN) if (TP + FN) > 0 else 0
    especificidade = TN / (TN + FP) if (TN + FP) > 0 else 0
    f1 = 2 * (precisao * revocacao) / (precisao + revocacao) if (precisao + revocacao) > 0 else 0

    print(f'Acurácia: {acuracia:.4f}')
    print(f'Precisão: {precisao:.4f}')
    print(f'Revocação (Recall): {revocacao:.4f}')
    print(f'Especificidade: {especificidade:.4f}')
    print(f'F1-Score: {f1:.4f}')
    print(f'TN: {TN}, FP: {FP}, FN: {FN}, TP: {TP}')
else:
    print("A matriz de confusão média não é 2x2. Não é possível calcular as métricas binárias.")


## 10. Análise dos Resultados e Métricas de Avaliação

Discussão dos resultados, comparação com o dataset original, explicação das métricas (acurácia, precisão, recall, especificidade, F1-score) e interpretação da matriz de confusão média.

## Visualização dos Agrupamentos com PCA

Para visualizar os agrupamentos do KMeans em 2D, utilizamos o PCA (Análise de Componentes Principais), que reduz a dimensionalidade dos dados mantendo a maior variância possível. Assim, é possível observar a separação dos clusters e a posição dos centros no espaço reduzido.

### Detalhes da Implementação

A implementação da visualização com PCA é realizada em algumas etapas principais:

1. **Padronização dos Dados**: Antes de aplicar o PCA, os dados são padronizados para que tenham média zero e variância um. Isso é importante porque o PCA é sensível à escala dos dados.

2. **Cálculo da Matriz de Covariância**: Em seguida, é calculada a matriz de covariância dos dados padronizados. Essa matriz descreve como as variáveis do conjunto de dados variam juntas.

3. **Cálculo dos Autovalores e Autovetores**: Os autovalores e autovetores da matriz de covariância são então calculados. Os autovalores indicam a quantidade de variância capturada por cada componente principal, enquanto os autovetores indicam a direção dos componentes.

4. **Seleção dos Principais Componentes**: Os principais componentes são selecionados com base em seus autovalores. Normalmente, escolhemos os componentes que explicam a maior parte da variância.

5. **Transformação dos Dados**: Finalmente, os dados são transformados para o novo espaço definido pelos principais componentes, resultando em um conjunto de dados de menor dimensão que ainda retém a estrutura original dos dados.

### Visualização

A visualização é feita em um gráfico de dispersão, onde cada ponto representa uma observação e está colorido de acordo com o seu respectivo cluster. Os eixos do gráfico representam os dois principais componentes selecionados pelo PCA. Essa visualização permite uma interpretação fácil da separação e agrupamento dos dados.

### Exemplo de Código

Aqui está um exemplo de como a visualização com PCA pode ser implementada em Python:

```python
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Supondo que 'dados' seja o seu conjunto de dados e 'kmeans' o modelo KMeans ajustado

# Padronização dos dados
dados_padronizados = StandardScaler().fit_transform(dados)

# Cálculo do PCA
pca = PCA(n_components=2)
componentes_principais = pca.fit_transform(dados_padronizados)

# Transformação dos dados
dados_transformados = pd.DataFrame(data=componentes_principais, columns=['Componente 1', 'Componente 2'])

# Visualização
plt.figure(figsize=(8, 6))
plt.scatter(dados_transformados['Componente 1'], dados_transformados['Componente 2'], c=kmeans.labels_, cmap='viridis')
plt.xlabel('Componente 1')
plt.ylabel('Componente 2')
plt.title('Visualização dos Agrupamentos com PCA')
plt.show()
```

Esse código realiza a padronização dos dados, calcula o PCA, transforma os dados para o novo espaço e, por fim, gera um gráfico de dispersão para a visualização dos agrupamentos.

**Como funciona o PCA:**

O PCA encontra as direções (componentes principais) que mais explicam a variância dos dados. Os dados são projetados nesses novos eixos, permitindo reduzir de dezenas de dimensões para 2, facilitando a visualização dos agrupamentos. No gráfico acima, cada ponto representa uma amostra projetada no novo espaço 2D, colorida conforme o cluster atribuído pelo KMeans. Os marcadores vermelhos (X) indicam os centros dos clusters nesse espaço reduzido.

## Análise dos Resultados

Compare os resultados obtidos com o dataset balanceado e o original. Observe se houve melhora na matriz de confusão, acurácia média e estabilidade do método.

## Métricas de Avaliação

- **Acurácia:** Proporção de acertos (positivos e negativos) sobre o total de exemplos. Mede o desempenho geral do modelo.
- **Precisão (Precision):** Proporção de positivos previstos que realmente são positivos. Mede a confiabilidade das previsões positivas.
- **Revocação (Recall) ou Sensibilidade:** Proporção de positivos reais que foram corretamente identificados. Mede a capacidade do modelo de encontrar todos os positivos.
- **Especificidade:** Proporção de negativos reais corretamente identificados. Mede a capacidade do modelo de identificar corretamente os negativos.
- **F1-Score:** Média harmônica entre precisão e revocação. Útil quando há desbalanceamento entre as classes.

Essas métricas são extraídas da matriz de confusão média das execuções repetidas.

In [ ]:
# Cálculo e exibição das principais métricas de avaliação (binária)
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

y_true_all = []
y_pred_all = []
for seed in range(1, 31):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = KMeansSupervisionado(n_clusters=n_clusters, random_state=seed)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)

acc = accuracy_score(y_true_all, y_pred_all)
prec = precision_score(y_true_all, y_pred_all, average='binary')
rec = recall_score(y_true_all, y_pred_all, average='binary')
f1 = f1_score(y_true_all, y_pred_all, average='binary')
cm = confusion_matrix(y_true_all, y_pred_all)
specificity = cm[0,0] / (cm[0,0] + cm[0,1]) if (cm[0,0] + cm[0,1]) > 0 else 0

print(f"Acurácia global: {acc:.4f}")
print(f"Precisão: {prec:.4f}")
print(f"Recall (Sensibilidade): {rec:.4f}")
print(f"Especificidade: {specificity:.4f}")
print(f"F1-score: {f1:.4f}")
print("Matriz de Confusão Média:")
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title('Matriz de Confusão Média - KMeans (Adult Balanceado)')
plt.show()